# 1113 51일차

## 1. 스케줄링 모드

CPU가 하는 배치 준비와 NPU가 하는 추론 연산을 겹쳐 돌리는 서버 옵션 (`f30_overlap_eager1`, `f30_overlap_eager2`)

기본값인 eager는 현재 배치를 끝낸 뒤에 다음 배치를 준비해서, 준비하는 동안 NPU의 연산 여력이 남음. 겹쳐 두면 그 구간이 줄어듦

```
furiosa-llm serve furiosa-ai/Qwen3-8B-FP8 --devices npu:6 --host 0.0.0.0 --port 8000 --enable-overlap-scheduling
```

1. Eager Scheduling : 기본값. 현재 배치를 끝낸 뒤 다음 배치를 준비함
2. Overlap Scheduling : NPU가 현재 배치를 처리하는 동안 CPU가 다음 배치를 준비함
3. `--enable-overlap-scheduling` : 서버를 띄울 때 붙여서 모드를 바꿈
4. `Eager scheduler has started with:` : 서버 로그에 지금 쓰는 스케줄러가 찍힘
5. 같은 스크립트를 두 모드에서 돌려 비교함 : 순차 요청(`f30_overlap_eager1`), 동시 요청 수별 비교(`f30_overlap_eager2`)

- **클라이언트 코드는 그대로 두고 서버를 띄우는 명령만 바꿈**
- 모드는 서버를 다시 띄워야 바뀌므로, 비교하려면 같은 스크립트를 두 번 돌림
- 출력 토큰 수가 적은 요청에서는 두 모드의 TPS 차이가 거의 없었음
- 같은 모델·같은 요청으로 맞춰야 모드 차이만 봄
- 동시 요청 수를 올렸을 때 올라가는 폭이 스케줄링 모드를 바꿀 때보다 훨씬 큼
- `--help`에는 처리량과 TPOT가 좋아지는 대신 TTFT가 조금 늘어난다고 적혀 있음
- speculative decoding과는 같이 쓸 수 없음

#### cf) 모드 바꿔 띄우기

- 쓰던 서버를 내리지 않고 같은 장치로 다시 띄우면 `EBUSY: Device or resource busy`가 남
- `ps -eo pid,etime,cmd | grep furiosa-llm`으로 장치를 잡고 있는 프로세스를 찾아 `kill`함
- `furiosa-smi status`의 메모리가 내려간 것을 보고 다시 띄움
- **NPU 하나를 한 서버가 통째로 잡으므로 두 모드를 동시에 띄워서 견줄 수는 없음**